# Build Evaluation Summaries

Utility notebook for consolidating CHAIR and POPE evaluation metrics into summary JSONL files that downstream tooling can consume.

In [1]:
import json
from pathlib import Path
from typing import Dict, List, Optional, Tuple


def resolve_repo_root() -> Path:
    start = Path.cwd().resolve()
    for path in [start, *start.parents]:
        opera_dir = path / "opera_log"
        scripts_dir = path / "scripts"
        if opera_dir.exists() and scripts_dir.exists():
            if (opera_dir / "chair_eval_results").exists() or (opera_dir / "pope_eval_results").exists():
                return path
    raise FileNotFoundError("Cannot locate repository root from current working directory.")


def parse_method_variant(name: str) -> Tuple[str, str]:
    if "-" in name:
        base, variant = name.split("-", 1)
    else:
        base, variant = name, "baseline"
    return base, variant


def canonical_metric_variant(value: Optional[str]) -> Optional[str]:
    if not value:
        return None
    normalized = value.replace("_", "-").lower()
    if normalized == "default":
        return "default"
    if normalized.startswith("alpha"):
        return "alpha-schedule"
    if normalized.startswith("improv"):
        return "improved"
    return None


_METRIC_SUFFIX_HINTS = [
    ("alpha-schedule", "alpha-schedule"),
    ("alpha_schedule", "alpha-schedule"),
    ("improved", "improved"),
    ("improv", "improved"),
]


def strip_metric_suffix(name: str) -> Tuple[str, Optional[str]]:
    normalized = name.lower()
    for raw, canonical in _METRIC_SUFFIX_HINTS:
        suffix = f"-{raw}"
        if normalized.endswith(suffix):
            trimmed = name[: -len(suffix)].rstrip("-_")
            return trimmed or name, canonical
    return name, None


def parse_metric_variant(stem: str, hint: Optional[str] = None) -> str:
    normalized = stem.lower()
    canonical_hint = canonical_metric_variant(hint)
    if normalized in {"metric", "metrics"}:
        return canonical_hint or "default"
    if normalized.endswith("_metric_improved"):
        return "improved"
    if normalized.endswith("_metric_alpha_schedule") or normalized.endswith("_metric_alpha-schedule"):
        return "alpha-schedule"
    if normalized.endswith("_metric"):
        return canonical_hint or "default"
    for prefix in ("metrics_", "metric_"):
        if normalized.startswith(prefix):
            suffix = stem[len(prefix):].lstrip("_-")
            canonical_suffix = canonical_metric_variant(suffix)
            if canonical_suffix:
                return canonical_suffix
    canonical_suffix = canonical_metric_variant(stem)
    if canonical_suffix:
        return canonical_suffix
    return canonical_hint or "default"


In [2]:
def collect_chair_records(chair_root: Path) -> List[Dict]:
    if not chair_root.exists():
        raise FileNotFoundError(f"Missing CHAIR results directory: {chair_root}")
    records: List[Dict] = []
    for provider_dir in sorted(p for p in chair_root.iterdir() if p.is_dir()):
        provider = provider_dir.name
        for model_dir in sorted(p for p in provider_dir.iterdir() if p.is_dir()):
            model = model_dir.name
            for method_dir in sorted(p for p in model_dir.iterdir() if p.is_dir()):
                method_label = method_dir.name
                method_core, method_metric_hint = strip_metric_suffix(method_label)
                method_base, method_variant = parse_method_variant(method_core)
                for metric_path in sorted(method_dir.glob("metric*.json")):
                    metric_variant = parse_metric_variant(metric_path.stem, hint=method_metric_hint)
                    try:
                        data = json.loads(metric_path.read_text())
                    except json.JSONDecodeError as exc:
                        raise ValueError(f"Failed parsing {metric_path}") from exc
                    overall = data.get("overall_metrics") or {
                        k: v for k, v in data.items() if isinstance(v, (int, float))
                    }
                    record = {
                        "dataset": "chair",
                        "provider": provider,
                        "model": model,
                        "method": method_base,
                        "variant": method_variant,
                        "metric_variant": metric_variant,
                        "source": str(metric_path.relative_to(chair_root.parent.parent)),
                        **overall,
                    }
                    records.append(record)
    return records


def _confusion_to_counts(matrix):
    try:
        (tp, fp), (fn, tn) = matrix
        total = tp + fp + fn + tn
        return tp, fp, fn, tn, total
    except (TypeError, ValueError):
        return None


def collect_pope_records(pope_root: Path) -> List[Dict]:
    if not pope_root.exists():
        raise FileNotFoundError(f"Missing POPE results directory: {pope_root}")
    type_alias_map = {"adv": "adversarial", "pop": "popular", "rand": "random"}
    records: List[Dict] = []
    metric_globs = ("*metric*.json", "*metric*.jsonl")
    for provider_dir in sorted(p for p in pope_root.iterdir() if p.is_dir()):
        provider = provider_dir.name
        for model_dir in sorted(p for p in provider_dir.iterdir() if p.is_dir()):
            model = model_dir.name
            for type_dir in sorted(p for p in model_dir.iterdir() if p.is_dir()):
                type_alias = type_dir.name
                pope_type = type_alias_map.get(type_alias)
                if pope_type is None:
                    continue
                for method_dir in sorted(p for p in type_dir.iterdir() if p.is_dir()):
                    method_label = method_dir.name
                    method_core, method_metric_hint = strip_metric_suffix(method_label)
                    method_base, method_variant = parse_method_variant(method_core)
                    metric_files: List[Path] = []
                    for pattern in metric_globs:
                        metric_files.extend(method_dir.glob(pattern))
                    for metric_path in sorted(metric_files):
                        try:
                            data = json.loads(metric_path.read_text())
                        except json.JSONDecodeError as exc:
                            raise ValueError(f"Failed parsing {metric_path}") from exc
                        matrix_counts = _confusion_to_counts(data.get("ConfusionMatrix"))
                        tp = data.get("TP")
                        fp = data.get("FP")
                        fn = data.get("FN")
                        tn = data.get("TN")
                        samples = data.get("TotalSamples")
                        if matrix_counts:
                            inferred_tp, inferred_fp, inferred_fn, inferred_tn, inferred_total = matrix_counts
                            tp = inferred_tp if tp is None else tp
                            fp = inferred_fp if fp is None else fp
                            fn = inferred_fn if fn is None else fn
                            tn = inferred_tn if tn is None else tn
                            samples = inferred_total if samples is None else samples
                        report = data.get("Report", {}) if isinstance(data.get("Report"), dict) else {}
                        macro = report.get("macro avg", {}) if isinstance(report, dict) else {}
                        weighted = report.get("weighted avg", {}) if isinstance(report, dict) else {}
                        metric_variant = parse_metric_variant(metric_path.stem, hint=method_metric_hint)
                        record = {
                            "dataset": "pope",
                            "provider": provider,
                            "model": model,
                            "pope_type": data.get("POPE_Type", pope_type),
                            "type_alias": type_alias,
                            "method": method_base,
                            "variant": method_variant,
                            "metric_variant": metric_variant,
                            "source": str(metric_path.relative_to(pope_root.parent.parent)),
                            "accuracy": data.get("Accuracy"),
                            "tp": tp,
                            "fp": fp,
                            "fn": fn,
                            "tn": tn,
                            "samples": samples,
                            "f1_macro": macro.get("f1-score"),
                            "precision_macro": macro.get("precision"),
                            "recall_macro": macro.get("recall"),
                            "f1_weighted": weighted.get("f1-score"),
                            "precision_weighted": weighted.get("precision"),
                            "recall_weighted": weighted.get("recall"),
                        }
                        records.append(record)
    return records


In [3]:
if __name__ == "__main__":
    REPO_ROOT = resolve_repo_root()
    CHAIR_ROOT = REPO_ROOT / "opera_log" / "chair_eval_results"
    POPE_ROOT = REPO_ROOT / "opera_log" / "pope_eval_results"
    SUMMARY_DIR = REPO_ROOT / "opera_log" / "summary_jsonl"
    SUMMARY_DIR.mkdir(parents=True, exist_ok=True)

    chair_records = collect_chair_records(CHAIR_ROOT)
    pope_records = collect_pope_records(POPE_ROOT)

    chair_records.sort(key=lambda r: (r["provider"], r["model"], r["method"], r["variant"], r["metric_variant"], r["source"]))
    pope_records.sort(key=lambda r: (r["provider"], r["model"], r.get("pope_type"), r["method"], r["variant"], r["source"]))

    def write_jsonl(path: Path, records):
        with path.open("w") as f:
            for record in records:
                f.write(json.dumps(record))
                f.write("\n")

    chair_path = SUMMARY_DIR / "chair_summary.jsonl"
    pope_path = SUMMARY_DIR / "pope_summary.jsonl"

    write_jsonl(chair_path, chair_records)
    write_jsonl(pope_path, pope_records)

    print(f"✅ Saved {len(chair_records)} CHAIR runs → {chair_path.relative_to(REPO_ROOT)}")
    print(f"✅ Saved {len(pope_records)} POPE runs → {pope_path.relative_to(REPO_ROOT)}")


✅ Saved 31 CHAIR runs → opera_log/summary_jsonl/chair_summary.jsonl
✅ Saved 57 POPE runs → opera_log/summary_jsonl/pope_summary.jsonl
